# IWTC Tools: Incremental Rebuild

This notebook provides a fast, deterministic rebuild path for canonical index and graph artifacts for a single world repository.

It replaces full pipeline execution with targeted rebuilds based on simple change flags, writing all outputs directly to the canonical index directory.

### Design Reference
- indexing_system_overview.md

### Repository Descriptor
- world_repository.yml

## Phase P0: Parameters

Define which world repository this notebook operates on and which index version it expects.

This notebook rebuilds index and graph data files. 

**IMPORTANT:** Unlike the bootstraping notebooks, this notebook writes directly to the canonical index directory identified in the descriptor file under indexes.path.

In [1]:
# Phase P0: Parameters
LAST_PHASE_RUN = "P0"

# Absolute path to the world_repository.yml descriptor.
WORLD_REPOSITORY_DESCRIPTOR = (
    "/Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/descriptors/world_repository.yml"
)

# Index version to load (must match previously generated artifacts)
INDEX_VERSION = "V0"
INDEX_VERSION_SUFFIX = INDEX_VERSION.lower()

# Indicate which inputs have changed
CHANGED_SOURCES        = True
CHANGED_ENTITIES       = True
CHANGED_ALIASES        = True
CHANGED_PREDICATES     = True
CHANGED_RELATIONSHIPS  = True

# Optional: Force Rebuild
FORCE_REBUILD_SOURCES  = False
FORCE_REBUILD_EVIDENCE = False
FORCE_REBUILD_SEMANTIC = False

# Internal run metadata (do not edit)
from datetime import datetime
print(f"Notebook run initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
del datetime

print("\nChanged inputs:")
print(f"{'Sources:':16} {CHANGED_SOURCES}")
print(f"{'Entities:':16} {CHANGED_ENTITIES}")
print(f"{'Aliases:':16} {CHANGED_ALIASES}")
print(f"{'Predicates:':16} {CHANGED_PREDICATES}")
print(f"{'Relationships:':16} {CHANGED_RELATIONSHIPS}")

print("\nForce rebuild:")
print(f"{'Raw sources:':16} {FORCE_REBUILD_SOURCES}")
print(f"{'Evidence graph:':16} {FORCE_REBUILD_EVIDENCE}")
print(f"{'Semantic graph:':16} {FORCE_REBUILD_SEMANTIC}")

Notebook run initialized at: 2026-03-23 19:26

Changed inputs:
Sources:         True
Entities:        True
Aliases:         True
Predicates:      True
Relationships:   True

Force rebuild:
Raw sources:     False
Evidence graph:  False
Semantic graph:  False


## Phase P1: Descriptor Load and Path Resolution

Load the world repository descriptor and resolve all required paths.

Fails fast if required fields are missing.

In [2]:
# Phase P1: Descriptor Load + Path Resolution
LAST_PHASE_RUN = "P1"

from pathlib import Path
import yaml
from IPython.display import display

p1_errors = []


# Load descriptor
descriptor_path = Path(WORLD_REPOSITORY_DESCRIPTOR)

if descriptor_path.exists():
    with descriptor_path.open("r", encoding="utf-8") as f:
        descriptor = yaml.safe_load(f)
    del f

    if not isinstance(descriptor, dict):
        raise ValueError("Descriptor file did not load as a dictionary.")
else:
    raise FileNotFoundError(f"Descriptor not found: {WORLD_REPOSITORY_DESCRIPTOR}")

del descriptor_path


# Resolve world root
WORLD_ROOT = descriptor.get("world_root")

if WORLD_ROOT and Path(WORLD_ROOT).is_absolute():
    WORLD_ROOT = Path(WORLD_ROOT)
else:
    raise ValueError("Descriptor must define world_root as an absolute path.")


# Resolve indexes path
INDEXES_PATH = None
INDEXES_RELPATH = descriptor.get("indexes", {}).get("path")

if INDEXES_RELPATH:
    indexes_path = Path(INDEXES_RELPATH)

    if indexes_path.is_absolute():
        INDEXES_PATH = indexes_path
    else:
        INDEXES_PATH = WORLD_ROOT / indexes_path

    if not INDEXES_PATH.exists():
        p1_errors.append(f"Indexes path not found: {INDEXES_RELPATH}")

    del indexes_path
else:
    p1_errors.append("Descriptor missing indexes.path")


# Resolve vocabulary paths
VOCAB_ENTITIES_PATH = None
VOCAB_ENTITIES_RELPATH = None
VOCAB_ALIASES_PATH = None
VOCAB_ALIASES_RELPATH = None
VOCAB_RELATIONSHIPS_PATH = None
VOCAB_RELATIONSHIPS_RELPATH = None
VOCAB_PREDICATES_PATH = None
VOCAB_PREDICATES_RELPATH = None

def resolve_vocab_path(key):
    relpath = descriptor.get("vocabulary", {}).get(key)

    if relpath:
        path_obj = Path(relpath)

        if path_obj.is_absolute():
            abspath = path_obj
        else:
            abspath = WORLD_ROOT / path_obj

        if not abspath.exists():
            p1_errors.append(f"Vocabulary file not found: {relpath}")

        del path_obj
        return abspath, relpath
    else:
        p1_errors.append(f"Descriptor missing vocabulary.{key}")
        return None, None

VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH = resolve_vocab_path("entities")
VOCAB_ALIASES_PATH, VOCAB_ALIASES_RELPATH = resolve_vocab_path("aliases")
VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH = resolve_vocab_path("relationships")
VOCAB_PREDICATES_PATH, VOCAB_PREDICATES_RELPATH = resolve_vocab_path("predicates")


# Resolve source paths (no loading yet)
SOURCE_PATHS = []
read_paths = descriptor.get("sources", {}).get("read_paths", [])

if isinstance(read_paths, list) and len(read_paths) > 0:
    for entry in read_paths:
        if isinstance(entry, dict):
            relpath = entry.get("path")
            source_type = entry.get("type")
        else:
            relpath = entry
            source_type = None

        if relpath:
            path_obj = Path(relpath)

            if path_obj.is_absolute():
                abspath = path_obj
            else:
                abspath = WORLD_ROOT / path_obj

            SOURCE_PATHS.append({
                "path": abspath,
                "relpath": relpath,
                "type": source_type,
            })

            if not abspath.exists():
                p1_errors.append(f"Source path not found: {relpath}")

            del path_obj, abspath
        else:
            p1_errors.append("A sources.read_paths entry is missing its path value.")

        del entry, relpath, source_type
else:
    p1_errors.append("Descriptor missing sources.read_paths or it is empty.")

if isinstance(read_paths, list) and len(read_paths) > 0 and len(SOURCE_PATHS) == 0:
    p1_errors.append("No valid source paths were resolved from sources.read_paths.")

del read_paths


# Raise once, with complete list
if p1_errors:
    print("Phase P1 failed. Please verify the descriptor file.\n")
    for msg in p1_errors:
        print(f"- {msg}")
    raise ValueError("Phase P1 failed. See error list above.")

del p1_errors, descriptor, yaml


# Summary
print("World Root:   ", WORLD_ROOT)
print("Indexes:      ", INDEXES_RELPATH)
print("Entities:     ", VOCAB_ENTITIES_RELPATH)
print("Aliases:      ", VOCAB_ALIASES_RELPATH)
print("Predicates:   ", VOCAB_PREDICATES_RELPATH)
print("Relationships:", VOCAB_RELATIONSHIPS_RELPATH)

print("\nSources:", len(SOURCE_PATHS), "read paths resolved")
display([s["relpath"] for s in SOURCE_PATHS])

print("\nPhase P1 OK: descriptor loaded and paths resolved.")

World Root:    /Users/charissophia/obsidian/Iron Wolf Trading Company
Indexes:       _meta/indexes
Entities:      _meta/indexes/vocab_entities.csv
Aliases:       _meta/indexes/vocab_aliases.csv
Predicates:    _meta/indexes/vocab_predicates.csv
Relationships: _meta/indexes/world_relationships.csv

Sources: 5 read paths resolved


['_local/auto_transcripts',
 '_local/pbp_transcripts',
 '_local/session_notes',
 '_local/planning_notes',
 '_local/recollections']


Phase P1 OK: descriptor loaded and paths resolved.


## Phase P2: Load Canonical Vocabulary Tables

Load the canonical vocabulary and semantic input tables from the index directory.

These files are small and may be reused across multiple later phases, so they are loaded once here and retained in memory.

Because these files are human-generated, column names are normalized here to a stable internal schema for later phases.

In [3]:
# Phase P2: Load and Normalize Canonical Vocabulary Tables
LAST_PHASE_RUN = "P2"

import pandas as pd
import re

p2_errors = []


# Allowed column variants
ENTITY_COLS = {
    "entity_id": ["entity_id", "id"],
    "canonical": ["canonical", "canonical_name", "name"],
}

ALIAS_COLS = {
    "entity_id": ["entity_id", "id"],
    "alias": ["alias", "alt", "alternate"],
}

PREDICATE_COLS = {
    "predicate": ["predicate"],
    "reverse_predicate": ["reverse_predicate", "reverse"],
    "include": ["include"],
    "relationship_class": ["relationship_class", "class"],
    "priority": ["priority", "weight", "score"],
    "cost": ["cost"],
}

RELATIONSHIP_COLS = {
    "subject_id": ["subject_id", "subject", "source_id", "from_id"],
    "predicate": ["predicate", "relationship", "relation", "edge", "verb"],
    "object_id": ["object_id", "object", "target_id", "to_id"],
}


def load_and_normalize_csv(path_obj, col_map, key_label):
    if path_obj.exists():
        raw_df = pd.read_csv(path_obj, dtype=str).fillna("")
    else:
        p2_errors.append(f"{key_label} file not found: {path_obj}")
        return None

    rename = {}
    for semantic, options in col_map.items():
        found = next((c for c in options if c in raw_df.columns), None)
        if found:
            rename[found] = semantic

    out = raw_df.rename(columns=rename)
    missing = [k for k in col_map.keys() if k not in out.columns]

    if len(missing) == 0:
        return out[list(col_map.keys())]

    p2_errors.append(f"[{key_label}] Missing required columns: {missing}")
    return None


# Load and normalize canonical vocab tables
DF_VOCAB_ENTITIES = load_and_normalize_csv(
    VOCAB_ENTITIES_PATH,
    ENTITY_COLS,
    "Entities",
)

DF_VOCAB_ALIASES = load_and_normalize_csv(
    VOCAB_ALIASES_PATH,
    ALIAS_COLS,
    "Aliases",
)

DF_VOCAB_PREDICATES = load_and_normalize_csv(
    VOCAB_PREDICATES_PATH,
    PREDICATE_COLS,
    "Predicates",
)

DF_VOCAB_RELATIONSHIPS = load_and_normalize_csv(
    VOCAB_RELATIONSHIPS_PATH,
    RELATIONSHIP_COLS,
    "Relationships",
)


# Normalize predicate data types
if DF_VOCAB_PREDICATES is not None:
    DF_VOCAB_PREDICATES["include"] = (
        DF_VOCAB_PREDICATES["include"]
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

    DF_VOCAB_PREDICATES["priority"] = pd.to_numeric(
        DF_VOCAB_PREDICATES["priority"],
        errors="coerce",
    )

    DF_VOCAB_PREDICATES["cost"] = pd.to_numeric(
        DF_VOCAB_PREDICATES["cost"],
        errors="coerce",
    )


# Build consolidated linking vocabulary (canonicals + aliases)
link_rows = []

for _, r in DF_VOCAB_ENTITIES.iterrows():
    if r["entity_id"] and r["canonical"]:
        link_rows.append(
            {
                "vocab": r["canonical"],
                "entity_id": r["entity_id"],
                "canonical": r["canonical"],
                "match_kind": "canonical",
            }
        )

canon_by_id = dict(
    zip(DF_VOCAB_ENTITIES["entity_id"], DF_VOCAB_ENTITIES["canonical"])
)

for _, r in DF_VOCAB_ALIASES.iterrows():
    if r["entity_id"] and r["alias"]:
        link_rows.append(
            {
                "vocab": r["alias"],
                "entity_id": r["entity_id"],
                "canonical": canon_by_id.get(r["entity_id"], ""),
                "match_kind": "alias",
            }
        )

DF_VOCAB_LINK = (
    pd.DataFrame(link_rows)
      .drop_duplicates(subset=["vocab", "entity_id"])
      .reset_index(drop=True)
)

if DF_VOCAB_LINK.empty:
    p2_errors.append("No vocabs available for linking.")
else:
    DF_VOCAB_LINK["vocab_len"] = DF_VOCAB_LINK["vocab"].str.len()
    DF_VOCAB_LINK = (
        DF_VOCAB_LINK
          .sort_values(["vocab_len", "vocab"], ascending=[False, True])
          .reset_index(drop=True)
    )

    patterns = []
    for vocab in DF_VOCAB_LINK["vocab"]:
        escaped = re.escape(vocab)
        escaped = escaped.replace(r"\ ", r"\s+")
        patterns.append(
            re.compile(rf"(?<!\w){escaped}(?!\w)", re.IGNORECASE)
        )
    DF_VOCAB_LINK["pattern"] = patterns
    del patterns, vocab, escaped


# Raise once, with complete list
if p2_errors:
    print("Phase P2 failed. Please verify the vocabulary files.\n")
    for msg in p2_errors:
        print(f"- {msg}")
    raise ValueError("Phase P2 failed. See error list above.")


# Summary
print(f"{'Entities:':14} {VOCAB_ENTITIES_RELPATH:40} ({len(DF_VOCAB_ENTITIES)} rows)")
print(f"{'Aliases:':14} {VOCAB_ALIASES_RELPATH:40} ({len(DF_VOCAB_ALIASES)} rows)")
print(f"{'Predicates:':14} {VOCAB_PREDICATES_RELPATH:40} ({len(DF_VOCAB_PREDICATES)} rows)")
print(f"{'Relationships:':14} {VOCAB_RELATIONSHIPS_RELPATH:40} ({len(DF_VOCAB_RELATIONSHIPS)} rows)")
print(f"{'Link vocabs:':14} {'(in memory)':40} ({len(DF_VOCAB_LINK)} rows)")
print()

display(DF_VOCAB_ENTITIES.head(3))
display(DF_VOCAB_ALIASES.head(3))
display(DF_VOCAB_PREDICATES.head(3))
display(DF_VOCAB_RELATIONSHIPS.head(3))
display(DF_VOCAB_LINK.head(10)[["vocab", "entity_id", "canonical", "match_kind", "vocab_len"]])

print("\nPhase P2 OK: canonical vocabulary tables loaded and normalized.")


# Clean up variables we won't need ongoing
del p2_errors, link_rows, canon_by_id, r
del ALIAS_COLS, ENTITY_COLS, PREDICATE_COLS, RELATIONSHIP_COLS
del VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH
del VOCAB_ALIASES_PATH, VOCAB_ALIASES_RELPATH
del VOCAB_PREDICATES_PATH, VOCAB_PREDICATES_RELPATH
del VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH

Entities:      _meta/indexes/vocab_entities.csv         (403 rows)
Aliases:       _meta/indexes/vocab_aliases.csv          (90 rows)
Predicates:    _meta/indexes/vocab_predicates.csv       (33 rows)
Relationships: _meta/indexes/world_relationships.csv    (312 rows)
Link vocabs:   (in memory)                              (486 rows)



,entity_id,canonical
0,art_folly,The Folly
1,art_glory,Unala's Glory
2,art_iris,IRIS


,entity_id,alias
0,art_folly,Folly
1,art_glory,Glory
2,pers_alivyre,Alivyre


,predicate,reverse_predicate,include,relationship_class,priority,cost
0,allied_with,allied_with,True,political,3,2
1,ambassador_to,has_ambassador,False,diplomatic,4,2
2,at_war_with,at_war_with,False,political,4,2


,subject_id,predicate,object_id
0,art_folly,belongs_to,org_folly
1,factn_burzath,current_member_of,org_orcs
2,factn_dakgorim,former_member_of,factn_ilguz


,vocab,entity_id,canonical,match_kind,vocab_len
0,Keeper of the Krach-ul Caves at Crafthold,off_lead_krach-ul_cavekeepers,Keeper of the Krach-ul Caves at Crafthold,canonical,41
1,Lead Handler of the Krach-ul at Crafthold,off_lead_krach-ul_handlers,Lead Handler of the Krach-ul at Crafthold,canonical,41
2,Krach-ul Animal Handlers at Crafthold,org_krach-ul_handlers,Krach-ul Animal Handlers at Crafthold,canonical,37
3,Krach-ul Cavekeepers at Crafthold,org_krach-ul_cavekeepers,Krach-ul Cavekeepers at Crafthold,canonical,33
4,Lieutenant of the Silver Jackals,off_dpty_jackals,Lieutenant of the Silver Jackals,canonical,32
5,Professor of the Elysian Academy,role_adm_academy_professor,Professor of the Elysian Academy,canonical,32
6,Stablemaster of Wolfstream Abbey,off_lead_wolfstream_stables,Stablemaster of Wolfstream Abbey,canonical,32
7,Hospitaller of Wolfstream Abbey,off_lead_wolfstream_hospitaller,Hospitaller of Wolfstream Abbey,canonical,31
8,Blacksmith of Wolfstream Abbey,off_ops_wolfstream_blacksmith,Blacksmith of Wolfstream Abbey,canonical,30
9,Crafthold Warehouse and Supply,org_ch_warehouse,Crafthold Warehouse and Supply,canonical,30



Phase P2 OK: canonical vocabulary tables loaded and normalized.


## Phase P3: Rebuild Source-Derived Indexes

Rebuild the source-derived entity index files if source content or source-matching vocabulary has changed.

Note that only .md and .txt files are currently loaded.

This phase writes directly to the canonical index directory.

In [6]:
# Phase P3: Rebuild Source-Derived Indexes
LAST_PHASE_RUN = "P3"

if (
    FORCE_REBUILD_SOURCES
    or CHANGED_SOURCES
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
):
    print("Phase P3 will run:")
    print(f"{'  Force rebuild sources:':24} {FORCE_REBUILD_SOURCES}")
    print(f"{'  Changed sources:':24} {CHANGED_SOURCES}")
    print(f"{'  Changed entities:':24} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':24} {CHANGED_ALIASES}")

    # ----------------------
    # Discover source files
    # ----------------------
    SOURCE_FILES = []

    for source in SOURCE_PATHS:
        if source["path"].exists() and source["path"].is_dir():
            for p in source["path"].rglob("*"):
                if p.is_file() and p.suffix.lower() in {".md", ".txt"}:
                    SOURCE_FILES.append({
                        "path": p,
                        "relpath": str(p.relative_to(WORLD_ROOT)),
                        "source_type": source["type"],
                    })
        else:
            if source["path"].is_file() and source["path"].suffix.lower() in {".md", ".txt"}:
                SOURCE_FILES.append({
                    "path": source["path"],
                    "relpath": str(source["path"].relative_to(WORLD_ROOT)),
                    "source_type": source["type"],
                })

    print(f"\nSource files discovered: {len(SOURCE_FILES)}")
    display([s["relpath"] for s in SOURCE_FILES[:10]])
    if len(SOURCE_FILES) > 10:
        print("  ... (truncated)")

    print("\nBy source type:")
    print(
        "  "
        + pd.Series([s["source_type"] or "untyped" for s in SOURCE_FILES])
            .value_counts().to_string().replace("\n", "\n  ")
    )
    del p

    # ----------------------
    # Load source files into memory as raw lines
    # ----------------------
    LOADED_SOURCES = []
    
    for item in SOURCE_FILES:
        suffix = item["path"].suffix.lower()
    
        if suffix in (".md", ".txt"):
            text = item["path"].read_text(encoding="utf-8", errors="replace")
            lines = text.splitlines()
            del text
    
            LOADED_SOURCES.append({
                "path": item["path"],
                "relpath": item["relpath"],
                "source_type": item["source_type"],
                "lines": lines,
            })
    
            del lines
        else:
            raise ValueError(f"Unsupported file type: {item['path']}")
    
    
    # Summary
    print(f"\nLoaded sources: {len(LOADED_SOURCES)}")
    
    for s in LOADED_SOURCES[:10]:
        print(f"  {s['relpath']}  ({len(s['lines'])} lines)")
    
    if len(LOADED_SOURCES) > 10:
        print("  ... (truncated)")
        
    # Clean up
    del SOURCE_FILES, item, suffix, s
    
    # ----------------------
    # Consolidated header-driven chunking
    # ----------------------
    TIME_LIKE_REGEX = r"\d{1,2}:\d{2}(?::\d{2})?(?:\s*[AP]M)?"
    
    # Header regexes are evaluated in order; first match wins.
    HEADER_REGEXES = [
        ("auto_ts"   , re.compile(r"^\s*\d{1,2}:\d{2}(?::\d{2})?\s*$")),
        ("pbp_hash"  , re.compile(rf"^\s*(?:\d+\.\s*)?(?:[*-]\s*)?###\s+.*{TIME_LIKE_REGEX}.*$")),
        ("pbp_forum" , re.compile(rf"^\s*>?\s*\*\*.*{TIME_LIKE_REGEX}.*\*\*\s*$")),
        ("session"   , re.compile(r"^\s*(?:\d+\.\s*)?(?:[>#*_\-\s]+)?session\s+(?!notes\b)\S.*$", re.IGNORECASE)),
        ("md_heading", re.compile(r"^\s*(?:[*\-]\s*)?#{1,6}\s+\S.*$")),
    ]
    
    CHUNKS_V0 = []
    chunk_global_id = 1
    
    for src in LOADED_SOURCES:
        current_kind = "preamble"
        current_lines = []
        chunk_start_line = 1
    
        for idx, line in enumerate(src["lines"], start=1):
            matched_kind = next(
                (kind for kind, header_regex in HEADER_REGEXES if header_regex.match(line)),
                None,
            )
    
            if matched_kind:
                if current_lines:
                    CHUNKS_V0.append(
                        {
                            "chunk_id": chunk_global_id,
                            "source_type": src["source_type"],
                            "relpath": src["relpath"],
                            "start_line": chunk_start_line,
                            "end_line": idx - 1,
                            "header_kind": current_kind,
                            "lines": list(current_lines),
                        }
                    )
                    chunk_global_id += 1
    
                current_kind = matched_kind
                current_lines = [line]
                chunk_start_line = idx
            else:
                current_lines.append(line)
    
        if current_lines:
            CHUNKS_V0.append(
                {
                    "chunk_id": chunk_global_id,
                    "source_type": src["source_type"],
                    "relpath": src["relpath"],
                    "start_line": chunk_start_line,
                    "end_line": len(src["lines"]),
                    "header_kind": current_kind,
                    "lines": list(current_lines),
                }
            )
            chunk_global_id += 1
    
    
    print(f"\nChunks created: {len(CHUNKS_V0)} from {len(LOADED_SOURCES)} files.")
    
    print("\nBy header kind:")
    print(
        "  "
        + pd.Series([c["header_kind"] for c in CHUNKS_V0])
            .value_counts()
            .to_string()
            .replace("\n", "\n  ")
    )
    print()
    display(
        pd.DataFrame(CHUNKS_V0)[
            ["chunk_id", "relpath", "start_line", "end_line", "header_kind"]
        ].head(10)
    )
    
    del TIME_LIKE_REGEX, HEADER_REGEXES, chunk_global_id
    del src, current_kind, current_lines, chunk_start_line, idx, line, matched_kind 

    # ----------------------
    # Link entities to chunks
    # ----------------------

    # Internal run metadata (do not edit)
    from datetime import datetime
    print(f"Start entity to chunk linking: (expect 2-4 minutes runtime) {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    
    EXCLUDE_SOURCE_TYPES = {"auto_transcripts"}
    MAX_SNIPPET_CHARS = 220

    vocabs = DF_VOCAB_LINK["vocab"].tolist()
    entity_ids = DF_VOCAB_LINK["entity_id"].tolist()
    canonicals = DF_VOCAB_LINK["canonical"].tolist()
    match_kinds = DF_VOCAB_LINK["match_kind"].tolist()
    patterns = DF_VOCAB_LINK["pattern"].tolist()

    rows = []
    mention_id = 1

    for chunk in CHUNKS_V0:
        source_type = chunk.get("source_type", "unknown")

        if source_type not in EXCLUDE_SOURCE_TYPES:
            lines = chunk.get("lines", [])
            header_kind = chunk.get("header_kind")

            # Drop pbp/session header line (metadata, not narrative)
            if header_kind in {"pbp_hash", "pbp_forum", "session"} and lines:
                content_lines = lines[1:]
                content_start_line = (chunk.get("start_line") or 1) + 1
            else:
                content_lines = lines
                content_start_line = chunk.get("start_line") or 1

            concat_text = " ".join(" ".join(content_lines).split())

            if concat_text:
                if len(concat_text) > MAX_SNIPPET_CHARS:
                    snippet = concat_text[: MAX_SNIPPET_CHARS - 3] + "..."
                else:
                    snippet = concat_text

                # Longest-first matching with span masking
                consumed = [False] * len(concat_text)

                for vocab, entity_id, canonical, match_kind, pat in zip(
                    vocabs, entity_ids, canonicals, match_kinds, patterns
                ):
                    hits = [(m.start(), m.end()) for m in pat.finditer(concat_text)]

                    if hits:
                        kept = []

                        for a, b in hits:
                            if a >= 0 and b > a:
                                if not any(consumed[a:b]):
                                    kept.append((a, b))

                        if kept:
                            for a, b in kept:
                                for k in range(a, b):
                                    consumed[k] = True

                            rows.append(
                                {
                                    "mention_id": mention_id,
                                    "entity_id": entity_id,
                                    "canonical": canonical,
                                    "matched_vocab": vocab,
                                    "match_kind": match_kind,
                                    "match_count_in_chunk": len(kept),

                                    "chunk_id": chunk["chunk_id"],
                                    "source_type": chunk["source_type"],
                                    "relpath": chunk["relpath"],
                                    "chunk_start_line": chunk["start_line"],
                                    "chunk_end_line": chunk["end_line"],
                                    "header_kind": chunk["header_kind"],

                                    "content_start_line": content_start_line,
                                    "snippet": snippet,
                                }
                            )
                            mention_id += 1

    ENTITY_MENTIONS_V0 = pd.DataFrame(rows)

    print(
        f"\nChunks scanned for entity linking: "
        f"{sum(1 for c in CHUNKS_V0 if c.get('source_type') not in EXCLUDE_SOURCE_TYPES)}"
    )
    print(f"Entity mentions: {len(ENTITY_MENTIONS_V0)} rows")

    if len(ENTITY_MENTIONS_V0) > 0:
        display(ENTITY_MENTIONS_V0.head(10))

    del EXCLUDE_SOURCE_TYPES, MAX_SNIPPET_CHARS
    del vocabs, entity_ids, canonicals, match_kinds, patterns
    del rows, mention_id


    
    print(f"All P3 output is complete at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    del datetime
# ----------------------
# resume overall structure here
# ----------------------
else:
    print("Phase P3 skipped. No source-derived inputs changed.")

Phase P3 will run:
  Force rebuild sources: False
  Changed sources:       True
  Changed entities:      True
  Changed aliases:       True

Source files discovered: 130


['_local/auto_transcripts/iwtc session 103.txt',
 '_local/auto_transcripts/iwtc session 088.txt',
 '_local/auto_transcripts/iwtc session 077.txt',
 '_local/auto_transcripts/iwtc session 063.txt',
 '_local/auto_transcripts/iwtc session 062.txt',
 '_local/auto_transcripts/iwtc session 089.txt',
 '_local/auto_transcripts/iwtc session 102.txt',
 '_local/auto_transcripts/iwtc session 048.txt',
 '_local/auto_transcripts/iwtc session 060.txt',
 '_local/auto_transcripts/iwtc session 074.txt']

  ... (truncated)

By source type:
  auto_transcripts    105
  planning_notes       11
  pbp_transcripts       9
  session_notes         5

Loaded sources: 130
  _local/auto_transcripts/iwtc session 103.txt  (3425 lines)
  _local/auto_transcripts/iwtc session 088.txt  (3135 lines)
  _local/auto_transcripts/iwtc session 077.txt  (3515 lines)
  _local/auto_transcripts/iwtc session 063.txt  (3597 lines)
  _local/auto_transcripts/iwtc session 062.txt  (3683 lines)
  _local/auto_transcripts/iwtc session 089.txt  (3683 lines)
  _local/auto_transcripts/iwtc session 102.txt  (2555 lines)
  _local/auto_transcripts/iwtc session 048.txt  (4061 lines)
  _local/auto_transcripts/iwtc session 060.txt  (3193 lines)
  _local/auto_transcripts/iwtc session 074.txt  (4013 lines)
  ... (truncated)

Chunks created: 170131 from 130 files.

By header kind:
  auto_ts       168537
  pbp_hash         860
  md_heading       439
  session          179
  preamble         116



,chunk_id,relpath,start_line,end_line,header_kind
0,1,_local/auto_transcripts/iwtc session 103.txt,1,1,preamble
1,2,_local/auto_transcripts/iwtc session 103.txt,2,3,auto_ts
2,3,_local/auto_transcripts/iwtc session 103.txt,4,5,auto_ts
3,4,_local/auto_transcripts/iwtc session 103.txt,6,7,auto_ts
4,5,_local/auto_transcripts/iwtc session 103.txt,8,9,auto_ts
5,6,_local/auto_transcripts/iwtc session 103.txt,10,11,auto_ts
6,7,_local/auto_transcripts/iwtc session 103.txt,12,13,auto_ts
7,8,_local/auto_transcripts/iwtc session 103.txt,14,15,auto_ts
8,9,_local/auto_transcripts/iwtc session 103.txt,16,17,auto_ts
9,10,_local/auto_transcripts/iwtc session 103.txt,18,19,auto_ts


Start entity to chunk linking: (expect 2-4 minutes runtime) 2026-03-23 19:33

Chunks scanned for entity linking: 1486
Entity mentions: 5369 rows


,mention_id,entity_id,canonical,matched_vocab,match_kind,match_count_in_chunk,chunk_id,source_type,relpath,chunk_start_line,chunk_end_line,header_kind,content_start_line,snippet
0,1,play_bysickle,Bysickle,Bysickle,canonical,1,168647,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
1,2,pers_ronric,Ronric,Ronric,canonical,5,168647,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
2,3,pers_victor,Victor D Evernight,Victor,alias,5,168647,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
3,4,pers_henry,Henry Sleepsong,Henry,alias,1,168647,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
4,5,pers_light,Light of Ironfang,Light,alias,1,168647,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
5,6,pers_gina,Gina,Gina,canonical,4,168647,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
6,7,play_crowe,Crowe,CroweTheDualityKing,alias,1,168648,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...
7,8,play_kalina,Kalina,Kalina Hitana,alias,1,168648,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...
8,9,creat_little_bear,Little Bear,Little Bear,canonical,1,168648,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...
9,10,org_academy_elysia,Elysian Academy,Academy,alias,1,168648,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...


All P3 output is complete at 2026-03-23 19:35


##